# Discovery Refresh

This notebook runs the shared discovery cache refresh used by the automated workflows.

It refreshes:
1. `data/latest_discovery.json`
2. `data/discovery_history.csv`

The heavy discovery logic lives in `run_discovery_cache.py`, so this notebook stays aligned with production.

In [ ]:
from pathlib import Path
import os
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "run_trade_from_cache.py").exists() or (candidate / "run_discovery_cache.py").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root from the current working directory.")

repo_root = find_repo_root(Path.cwd())
os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f"Repository root: {repo_root}")

In [ ]:
import os

os.environ['DISCOVERY_CACHE_PATH'] = 'data/latest_discovery.json'

module_name = 'run_discovery_cache'
try:
    module = __import__(module_name, fromlist=["main"])
except ModuleNotFoundError as exc:
    print(f"Runner dependency missing: {exc}")
    module = None

if module is not None:
    try:
        module.main()
    except SystemExit as exc:
        if exc.code not in (0, None):
            raise
        print(f"Runner exited cleanly with code {exc.code}.")